<a href="https://colab.research.google.com/github/shahabhere925/AI-LAB-ASSIGNMENT-/blob/main/AIlabproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:



# AI-Based Network Pathfinding & Attack Simulation System



import heapq
import time
from collections import deque

# ============================================================
# SECTION 1: NETWORK TOPOLOGY (Main Graph - 10 nodes)
# ============================================================

GRAPH = {
    'Entry_Point':   [('Workstation_A', 2), ('Firewall_1', 5)],
    'Workstation_A': [('Entry_Point', 2), ('Server_1', 4), ('Workstation_B', 1)],
    'Workstation_B': [('Workstation_A', 1), ('Server_1', 3), ('Database_1', 7)],
    'Firewall_1':    [('Entry_Point', 5), ('Server_2', 2), ('DMZ_Node', 3)],
    'Server_1':      [('Workstation_A', 4), ('Workstation_B', 3), ('Database_1', 2)],
    'Server_2':      [('Firewall_1', 2), ('DMZ_Node', 1), ('Database_2', 6)],
    'DMZ_Node':      [('Firewall_1', 3), ('Server_2', 1), ('Database_2', 4)],
    'Database_1':    [('Workstation_B', 7), ('Server_1', 2), ('Target_System', 3)],
    'Database_2':    [('Server_2', 6), ('DMZ_Node', 4), ('Target_System', 2)],
    'Target_System': [('Database_1', 3), ('Database_2', 2)],
}

# Heuristic for A* (estimated cost to Target_System)
HEURISTIC = {
    'Entry_Point':   10,
    'Workstation_A': 8,
    'Workstation_B': 7,
    'Firewall_1':    7,
    'Server_1':      5,
    'Server_2':      5,
    'DMZ_Node':      4,
    'Database_1':    3,
    'Database_2':    2,
    'Target_System': 0,
}

START = 'Entry_Point'
GOAL  = 'Target_System'

# ============================================================
# SECTION 2: BFS
# ============================================================

def bfs(graph, start, goal):
    start_time = time.time()
    queue = deque([[start]])
    visited = set([start])
    nodes_expanded = 0

    while queue:
        path = queue.popleft()
        node = path[-1]
        nodes_expanded += 1

        if node == goal:
            cost = path_cost(graph, path)
            return path, cost, nodes_expanded, (time.time() - start_time) * 1000

        for neighbor, _ in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])

    return None, float('inf'), nodes_expanded, (time.time() - start_time) * 1000

# ============================================================
# SECTION 3: DFS
# ============================================================

def dfs(graph, start, goal):
    start_time = time.time()
    stack = [[start]]
    visited = set()
    nodes_expanded = 0

    while stack:
        path = stack.pop()
        node = path[-1]

        if node in visited:
            continue
        visited.add(node)
        nodes_expanded += 1

        if node == goal:
            cost = path_cost(graph, path)
            return path, cost, nodes_expanded, (time.time() - start_time) * 1000

        for neighbor, _ in reversed(graph.get(node, [])):
            if neighbor not in visited:
                stack.append(path + [neighbor])

    return None, float('inf'), nodes_expanded, (time.time() - start_time) * 1000

# ============================================================
# SECTION 4: UCS
# ============================================================

def ucs(graph, start, goal):
    start_time = time.time()
    heap = [(0, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        cost, path = heapq.heappop(heap)
        node = path[-1]

        if node in visited and visited[node] <= cost:
            continue
        visited[node] = cost
        nodes_expanded += 1

        if node == goal:
            return path, cost, nodes_expanded, (time.time() - start_time) * 1000

        for neighbor, weight in graph.get(node, []):
            new_cost = cost + weight
            if neighbor not in visited or visited.get(neighbor, float('inf')) > new_cost:
                heapq.heappush(heap, (new_cost, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.time() - start_time) * 1000

# ============================================================
# SECTION 5: A* Search
# ============================================================

def a_star(graph, start, goal, heuristic):
    start_time = time.time()
    heap = [(heuristic[start], 0, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        f, g, path = heapq.heappop(heap)
        node = path[-1]

        if node in visited and visited[node] <= g:
            continue
        visited[node] = g
        nodes_expanded += 1

        if node == goal:
            return path, g, nodes_expanded, (time.time() - start_time) * 1000

        for neighbor, weight in graph.get(node, []):
            new_g = g + weight
            if neighbor not in visited or visited.get(neighbor, float('inf')) > new_g:
                new_f = new_g + heuristic.get(neighbor, 0)
                heapq.heappush(heap, (new_f, new_g, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.time() - start_time) * 1000

# ============================================================
# SECTION 6A: Hill Climbing — Normal Run (succeeds)
# ============================================================

def hill_climbing(graph, start, goal, heuristic):
    start_time = time.time()
    current = start
    path = [current]
    visited = set([current])
    nodes_expanded = 0

    while current != goal:
        neighbors = graph.get(current, [])
        nodes_expanded += 1

        candidates = []
        for n, _ in neighbors:
            if n not in visited:
                candidates.append((heuristic.get(n, float('inf')), n))

        if not candidates:
            print(f"  [Hill Climbing] STUCK at '{current}' — no unvisited neighbors!")
            cost = path_cost(graph, path)
            return path, cost, nodes_expanded, (time.time() - start_time) * 1000

        best_h, best_node = min(candidates)

        if best_h >= heuristic.get(current, float('inf')):
            print(f"  [Hill Climbing] STUCK at '{current}' — all neighbors are worse (local maximum)!")
            cost = path_cost(graph, path)
            return path, cost, nodes_expanded, (time.time() - start_time) * 1000

        visited.add(best_node)
        path.append(best_node)
        current = best_node

    cost = path_cost(graph, path)
    return path, cost, nodes_expanded, (time.time() - start_time) * 1000


# ============================================================
# SECTION 6B: Hill Climbing — LOCAL MAXIMA FAILURE DEMO
# ============================================================
# This specially designed graph has a "trap" node (Trap_Node)
# whose heuristic looks great from the start but has NO path
# forward to the goal — demonstrating a local maximum failure.

TRAP_GRAPH = {
    'Start':      [('Node_A', 1), ('Node_B', 1)],
    'Node_A':     [('Start', 1), ('Trap_Node', 1)],   # Trap looks attractive!
    'Node_B':     [('Start', 1), ('Node_C', 1)],
    'Trap_Node':  [('Node_A', 1)],                     # Dead end — no way out!
    'Node_C':     [('Node_B', 1), ('Goal', 1)],
    'Goal':       [('Node_C', 1)],
}

# Heuristic: Trap_Node has h=1 (looks very close!) but is a dead end
TRAP_HEURISTIC = {
    'Start':     5,
    'Node_A':    3,
    'Node_B':    4,
    'Trap_Node': 1,
    'Node_C':    2,
    'Goal':      0,
}

def hill_climbing_local_maxima_demo():
    print("\n" + "="*60)
    print("  HILL CLIMBING — LOCAL MAXIMA FAILURE DEMONSTRATION")
    print("="*60)
    print("  Graph Layout:")
    print("    Start → Node_A → Trap_Node (DEAD END, h=1)")
    print("    Start → Node_B → Node_C → Goal")
    print("  Problem: Trap_Node has h=1 (best-looking neighbor),")
    print("  but it's a dead end with no path to Goal!")
    print("="*60)

    start_time = time.time()
    current = 'Start'
    path = [current]
    visited = set([current])
    nodes_expanded = 0
    stuck = False

    while current != 'Goal':
        neighbors = TRAP_GRAPH.get(current, [])
        nodes_expanded += 1

        candidates = []
        for n, _ in neighbors:
            if n not in visited:
                candidates.append((TRAP_HEURISTIC.get(n, float('inf')), n))

        if not candidates:
            print(f"\n  *** ALGORITHM STUCK at '{current}' ***")
            print(f"  No unvisited neighbors available — COMPLETE FAILURE!")
            stuck = True
            break

        best_h, best_node = min(candidates)

        if best_h >= TRAP_HEURISTIC.get(current, float('inf')):
            print(f"\n  *** ALGORITHM STUCK at '{current}' ***")
            print(f"  All neighbors have worse heuristic — LOCAL MAXIMUM!")
            stuck = True
            break

        print(f"  Moving: {current} (h={TRAP_HEURISTIC[current]}) → {best_node} (h={TRAP_HEURISTIC.get(best_node,'?')})")
        visited.add(best_node)
        path.append(best_node)
        current = best_node

    t = (time.time() - start_time) * 1000
    cost = path_cost(TRAP_GRAPH, path)

    print(f"\n  Path Taken   : {' → '.join(path)}")
    print(f"  Goal Reached : {'NO — FAILED (Local Maximum)' if stuck else 'YES'}")
    print(f"  Nodes Exp.   : {nodes_expanded}")
    print(f"  Time (ms)    : {t:.4f}")
    print("="*60)
    print("  LESSON: Hill Climbing is GREEDY — it picks the best")
    print("  neighbor locally but cannot backtrack. A misleading")
    print("  heuristic leads it into a dead end it can't escape.")
    print("="*60)

# ============================================================
# SECTION 7: MINIMAX with Alpha-Beta Pruning
# ============================================================

def minimax(graph, node, depth, is_maximizer, goal, heuristic, visited=None):
    if visited is None:
        visited = set()
    visited = visited | {node}

    if node == goal:
        return heuristic.get(node, 0) + depth * 10, [node]
    if depth == 0:
        return heuristic.get(node, 0), [node]

    neighbors = [n for n, _ in graph.get(node, []) if n not in visited]
    if not neighbors:
        return heuristic.get(node, 0), [node]

    if is_maximizer:
        best_val = float('-inf')
        best_path = [node]
        for neighbor in neighbors:
            val, path = minimax(graph, neighbor, depth - 1, False, goal, heuristic, visited)
            if val > best_val:
                best_val = val
                best_path = [node] + path
        return best_val, best_path
    else:
        best_val = float('inf')
        best_path = [node]
        for neighbor in neighbors:
            val, path = minimax(graph, neighbor, depth - 1, True, goal, heuristic, visited)
            if val < best_val:
                best_val = val
                best_path = [node] + path
        return best_val, best_path


def alpha_beta(graph, node, depth, alpha, beta, is_maximizer, goal, heuristic, visited=None):
    if visited is None:
        visited = set()
    visited = visited | {node}

    if node == goal:
        return heuristic.get(node, 0) + depth * 10, [node]
    if depth == 0:
        return heuristic.get(node, 0), [node]

    neighbors = [n for n, _ in graph.get(node, []) if n not in visited]
    if not neighbors:
        return heuristic.get(node, 0), [node]

    if is_maximizer:
        best_val = float('-inf')
        best_path = [node]
        for neighbor in neighbors:
            val, path = alpha_beta(graph, neighbor, depth - 1, alpha, beta, False, goal, heuristic, visited)
            if val > best_val:
                best_val = val
                best_path = [node] + path
            alpha = max(alpha, best_val)
            if beta <= alpha:
                print(f"    [Alpha-Beta] Beta cutoff at '{neighbor}' — branch pruned!")
                break
        return best_val, best_path
    else:
        best_val = float('inf')
        best_path = [node]
        for neighbor in neighbors:
            val, path = alpha_beta(graph, neighbor, depth - 1, alpha, beta, True, goal, heuristic, visited)
            if val < best_val:
                best_val = val
                best_path = [node] + path
            beta = min(beta, best_val)
            if beta <= alpha:
                print(f"    [Alpha-Beta] Alpha cutoff at '{neighbor}' — branch pruned!")
                break
        return best_val, best_path

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def path_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        src, dst = path[i], path[i + 1]
        for neighbor, cost in graph.get(src, []):
            if neighbor == dst:
                total += cost
                break
    return total

def print_result(name, path, cost, nodes, time_ms):
    path_str = " → ".join(path) if path else "No path found"
    print(f"\n{'='*60}")
    print(f"  Algorithm  : {name}")
    print(f"  Path       : {path_str}")
    print(f"  Total Cost : {cost}")
    print(f"  Nodes Exp. : {nodes}")
    print(f"  Time (ms)  : {time_ms:.4f}")
    print(f"{'='*60}")

def print_table(results):
    print("\n" + "="*85)
    print(f"{'COMPARATIVE ANALYSIS TABLE':^85}")
    print("="*85)
    print(f"{'Algorithm':<20} {'Path Found':<12} {'Total Cost':<12} {'Nodes Expanded':<16} {'Time (ms)':<10}")
    print("-"*85)
    for name, path, cost, nodes, t in results:
        found = "Yes" if path else "No"
        cost_str = str(cost) if cost != float('inf') else "N/A"
        print(f"{name:<20} {found:<12} {cost_str:<12} {nodes:<16} {t:.4f}")
    print("="*85)

# ============================================================
# SECTION 8: MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "#"*60)
    print("  AI-Based Network Pathfinding & Attack Simulation System")
    print(f"  Start Node : {START}")
    print(f"  Goal Node  : {GOAL}")
    print("#"*60)

    results = []

    # BFS
    path, cost, nodes, t = bfs(GRAPH, START, GOAL)
    print_result("BFS", path, cost, nodes, t)
    results.append(("BFS", path, cost, nodes, t))

    # DFS
    path, cost, nodes, t = dfs(GRAPH, START, GOAL)
    print_result("DFS", path, cost, nodes, t)
    results.append(("DFS", path, cost, nodes, t))

    # UCS
    path, cost, nodes, t = ucs(GRAPH, START, GOAL)
    print_result("UCS", path, cost, nodes, t)
    results.append(("UCS", path, cost, nodes, t))

    # A*
    path, cost, nodes, t = a_star(GRAPH, START, GOAL, HEURISTIC)
    print_result("A*", path, cost, nodes, t)
    results.append(("A*", path, cost, nodes, t))

    # Hill Climbing — Normal
    print("\n[Hill Climbing — Normal Run on Main Graph]")
    path, cost, nodes, t = hill_climbing(GRAPH, START, GOAL, HEURISTIC)
    print_result("Hill Climbing", path, cost, nodes, t)
    results.append(("Hill Climbing", path, cost, nodes, t))

    # Hill Climbing — Local Maxima Demo
    hill_climbing_local_maxima_demo()

    # Minimax
    print("\n[Minimax — Adversarial: Attacker(MAX) vs Defender(MIN), depth=4]")
    start_time = time.time()
    val, path = minimax(GRAPH, START, 4, True, GOAL, HEURISTIC)
    t = (time.time() - start_time) * 1000
    cost = path_cost(GRAPH, path)
    print_result("Minimax", path, cost, len(path), t)
    results.append(("Minimax", path, cost, len(path), t))

    # Alpha-Beta
    print("\n[Alpha-Beta Pruning — Same as Minimax but with pruning optimization]")
    start_time = time.time()
    val, path = alpha_beta(GRAPH, START, 4, float('-inf'), float('inf'), True, GOAL, HEURISTIC)
    t = (time.time() - start_time) * 1000
    cost = path_cost(GRAPH, path)
    print_result("Alpha-Beta", path, cost, len(path), t)
    results.append(("Alpha-Beta", path, cost, len(path), t))

    # Final Comparison Table
    print_table(results)

    print("\n[NOTE] Hill Climbing local maxima demo uses a separate trap graph.")
    print("       See Section 6B above for full details and explanation.")
